# Token Erasure Analysis

Measures which input tokens matter most for predictions via erasure,
token necessity/sufficiency, and erasure curves.

**References:**
- Li et al. (2016) "Understanding Neural Networks through Representation Erasure"
- De Cao et al. (2020) "How Do Decisions Emerge across Layers in Neural Models?"

## Why This Matters

Token erasure measures the effect of removing each input token on the model's prediction, establishing necessity and sufficiency for individual tokens. Erasure curves and pairwise interactions reveal which tokens are essential and how they cooperate to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_erasure import (
    token_erasure_effects,
    token_necessity_sufficiency,
    erasure_curve,
    pairwise_token_interaction,
    layerwise_token_importance,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print(f"Model: {cfg.n_layers} layers, seq_len={len(tokens)}")

In [ ]:
# 1. Token erasure effects - which tokens matter most?
eff = token_erasure_effects(model, tokens, metric_fn)
print(f"Most important token: pos {eff['most_important_token']}")
print(f"Least important token: pos {eff['least_important_token']}")
print(f"Mean effect: {eff['mean_effect']:.4f}")
print(f"\nToken importance ranking:")
for rank, pos in enumerate(eff['importance_ranking']):
    bar = '#' * int(eff['erasure_effects'][pos] * 30 / (max(eff['erasure_effects']) + 1e-10))
    print(f"  #{rank+1} pos {pos} (tok {int(tokens[pos])}): {eff['erasure_effects'][pos]:.4f} {bar}")

In [ ]:
# 2. Token necessity and sufficiency
ns = token_necessity_sufficiency(model, tokens, metric_fn)
print(f"Necessary tokens: {ns['necessary_tokens']}")
print(f"Sufficient tokens: {ns['sufficient_tokens']}")
print(f"Both necessary AND sufficient: {ns['both_necessary_and_sufficient']}")
print(f"\nPer-token scores:")
for pos in range(len(tokens)):
    n_bar = '#' * int(ns['necessity_scores'][pos] * 20)
    s_bar = '#' * int(ns['sufficiency_scores'][pos] * 20)
    print(f"  pos {pos}: necessity={ns['necessity_scores'][pos]:.3f} {n_bar}")
    print(f"          sufficiency={ns['sufficiency_scores'][pos]:.3f} {s_bar}")

In [ ]:
# 3. Erasure curve - metric vs number of tokens erased
curve = erasure_curve(model, tokens, metric_fn)
print(f"AUC (normalized): {curve['area_under_curve']:.4f}")
print(f"Tokens for 50% drop: {curve['tokens_for_50pct_drop']}")
print(f"\nErasure curve (most important first):")
baseline = curve['metrics'][0]
for i in range(len(curve['metrics'])):
    pct = abs(curve['metrics'][i]) / (abs(baseline) + 1e-10) * 100
    bar = '#' * int(pct / 5)
    print(f"  {curve['n_erased'][i]} erased: {curve['metrics'][i]:+.4f} ({pct:.1f}%) {bar}")

In [ ]:
# 4. Pairwise token interaction
for a, b in [(0, 1), (0, -1), (1, 2)]:
    inter = pairwise_token_interaction(model, tokens, metric_fn, pos_a=a, pos_b=b)
    print(f"Tokens {a} & {b}: {inter['interaction_type']}")
    print(f"  effect_a={inter['effect_a']:.4f}, effect_b={inter['effect_b']:.4f}")
    print(f"  effect_both={inter['effect_both']:.4f}, interaction={inter['interaction']:+.4f}")

In [ ]:
# 5. Layerwise token importance - where does each token become important?
lw = layerwise_token_importance(model, tokens, metric_fn)
print(f"Emergence layers: {lw['emergence_layer'].tolist()}")
print(f"Peak layers: {lw['peak_layer'].tolist()}")
print(f"\nImportance matrix (layers x tokens):")
for l in range(cfg.n_layers):
    vals = ' '.join(f"{lw['importance_matrix'][l, p]:.3f}" for p in range(len(tokens)))
    print(f"  Layer {l}: {vals}")